# 06 - Train the ECG-CXR bridge

Only frozen ECG and CXR feature arrays are loaded. Same-row pairs from the Symile training split are positives. The validation loss selects the checkpoint.

In [ ]:
from pathlib import Path
import importlib
import json
import os
import random
import sys
import time

import numpy as np
import torch
from torch.nn.utils import clip_grad_norm_
from torch.utils.data import DataLoader, Dataset

ARTIFACT_ROOT = Path(os.environ.get(
    "ECG_CXR_BRIDGE_ARTIFACT_ROOT",
    "/path/to/ecg_cxr_bridge_artifacts",
))
FEATURE_ROOT = ARTIFACT_ROOT / "features"
CODE_ROOT = ARTIFACT_ROOT / "code"
if str(CODE_ROOT) not in sys.path:
    sys.path.insert(0, str(CODE_ROOT))
import ecg_cxr_codebind
importlib.reload(ecg_cxr_codebind)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
RUN_NAME = os.environ.get("ECG_CXR_RUN_NAME", "symile_ecg_cxr_bridge_v1")
SEED = int(os.environ.get("ECG_CXR_SEED", "43"))
BATCH_SIZE = int(os.environ.get("ECG_CXR_BATCH_SIZE", "256"))
NUM_WORKERS = int(os.environ.get("ECG_CXR_NUM_WORKERS", "4"))
EPOCHS = int(os.environ.get("ECG_CXR_EPOCHS", "50"))
LR = float(os.environ.get("ECG_CXR_LR", "3e-4"))
WEIGHT_DECAY = float(os.environ.get("ECG_CXR_WEIGHT_DECAY", "1e-4"))
PATIENCE = int(os.environ.get("ECG_CXR_PATIENCE", "8"))
USE_AMP = os.environ.get("ECG_CXR_AMP", "1") == "1" and DEVICE.type == "cuda"
MODEL_HPARAMS = {"common_dim": 256, "specific_dim": 256, "code_dim": 8,
                 "common_codebook_size": 512, "specific_codebook_size": 256,
                 "code_temperature": 0.10, "dropout": 0.10}
LOSS_WEIGHTS = {"temperature": 0.07, "alignment_weight": 1.0,
                "code_relation_weight": 0.10, "commitment_weight": 0.25,
                "orthogonality_weight": 0.05, "code_usage_weight": 0.01}
print({"run": RUN_NAME, "device": str(DEVICE), "model": MODEL_HPARAMS, "loss": LOSS_WEIGHTS})


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

class PairDataset(Dataset):
    def __init__(self, root, split):
        self.paths = {"ecg": Path(root) / split / "ecg_hubert.npy",
                      "cxr": Path(root) / split / "cxr_raddino.npy"}
        self.ids_path = Path(root) / split / "pair_ids.npz"
        missing = [str(path) for path in self.paths.values() if not path.exists()]
        if missing:
            raise FileNotFoundError("Run notebooks 03 and 04 first:\n" + "\n".join(missing))
        self.arrays = {name: np.load(path, mmap_mode="r") for name, path in self.paths.items()}
        manifest = [json.loads(line) for line in (ARTIFACT_ROOT / "manifests" / f"{split}.jsonl").read_text().splitlines()]
        self.subject_id = np.asarray([row["subject_id"] for row in manifest], dtype=np.int64)
        self.hadm_id = np.asarray([row["hadm_id"] for row in manifest], dtype=np.int64)
        assert all(len(array) == len(manifest) for array in self.arrays.values())

    def __len__(self):
        return len(self.subject_id)

    def __getitem__(self, index):
        return {"ecg": torch.from_numpy(np.array(self.arrays["ecg"][index], dtype="float32", copy=True)),
                "cxr": torch.from_numpy(np.array(self.arrays["cxr"][index], dtype="float32", copy=True)),
                "subject_id": torch.tensor(self.subject_id[index], dtype=torch.long),
                "hadm_id": torch.tensor(self.hadm_id[index], dtype=torch.long)}

train_dataset = PairDataset(FEATURE_ROOT, "train")
val_dataset = PairDataset(FEATURE_ROOT, "val")
print("Rows:", len(train_dataset), len(val_dataset))


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
                          num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                          persistent_workers=NUM_WORKERS > 0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == "cuda",
                        persistent_workers=NUM_WORKERS > 0)
sample = train_dataset[0]
input_dims = {name: int(sample[name].numel()) for name in ecg_cxr_codebind.MODALITIES}
model = ecg_cxr_codebind.ECGCXRCodeBind(input_dims, **MODEL_HPARAMS).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR / 50)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print("Input dimensions:", input_dims)
print("Trainable parameters:", sum(parameter.numel() for parameter in model.parameters()))


In [ ]:
def move(batch):
    return {key: value.to(DEVICE, non_blocking=True) if isinstance(value, torch.Tensor) else value
            for key, value in batch.items()}

def run_epoch(loader, training):
    model.train(training)
    rows = []
    context = torch.enable_grad if training else torch.no_grad
    with context():
        for batch in loader:
            batch = move(batch)
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                loss, components = ecg_cxr_codebind.bridge_loss(model(batch), batch, LOSS_WEIGHTS)
            if training:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            rows.append({name: float(value.detach().cpu()) for name, value in components.items()})
    return {name: float(np.mean([row[name] for row in rows])) for name in rows[0]}


In [ ]:
run_dir = ARTIFACT_ROOT / "models" / RUN_NAME
run_dir.mkdir(parents=True, exist_ok=True)
resolved = {"run_name": RUN_NAME, "seed": SEED, "batch_size": BATCH_SIZE,
            "epochs": EPOCHS, "learning_rate": LR, "weight_decay": WEIGHT_DECAY,
            "input_dims": input_dims, "model_hparams": MODEL_HPARAMS,
            "loss_weights": LOSS_WEIGHTS}
(run_dir / "resolved_config.json").write_text(json.dumps(resolved, indent=2))
best = float("inf")
stale = 0
history = []
for epoch in range(EPOCHS):
    started = time.time()
    train_metrics = run_epoch(train_loader, True)
    val_metrics = run_epoch(val_loader, False)
    scheduler.step()
    row = {"epoch": epoch, "seconds": time.time() - started,
           "learning_rate": scheduler.get_last_lr()[0],
           "train": train_metrics, "validation": val_metrics}
    history.append(row)
    print(json.dumps(row))
    with (run_dir / "metrics.jsonl").open("a") as stream:
        stream.write(json.dumps(row) + "\n")
    checkpoint = {"epoch": epoch, "model": model.state_dict(),
                  "optimizer": optimizer.state_dict(), "resolved_config": resolved}
    torch.save(checkpoint, run_dir / "last.pt")
    if val_metrics["total"] < best:
        best = val_metrics["total"]
        stale = 0
        torch.save(checkpoint, run_dir / "best.pt")
    else:
        stale += 1
        if stale >= PATIENCE:
            print("Early stopping at epoch", epoch)
            break
print("Best validation loss:", best, "Saved:", run_dir)
